# Calculator Unit Validation Demo

This notebook demonstrates the new unit validation system for Calculator elements in the system dynamics toolkit.

## Problem Solved

Previously, the calculator system did not validate units, allowing dimensionally inconsistent expressions to pass silently. This could lead to:
- Invalid unit combinations (e.g., adding people + people/year)
- Wrong declared units being ignored
- Silent calculation errors

## Solution

The new unit validation system:
- ✅ Validates dimensional consistency of expressions
- ✅ Detects incompatible unit operations
- ✅ Provides clear error messages
- ✅ Supports both warning and exception modes
- ✅ Works with YAML-loaded models

In [ ]:
import sys
sys.path.append('../sd_toolkit')

from sd_toolkit.engine.elements import Stock, Flow, Calculator
from sd_toolkit.config.builder import YAMLSystemBuilder
import warnings

## Example 1: Valid Unit Combinations

In [ ]:
# Create elements with compatible units
birth_rate = Flow('Birth_Rate', rate=25, units='people/year')
death_rate = Flow('Death_Rate', rate=15, units='people/year')

# Valid calculator: people/year - people/year = people/year
net_growth = Calculator(
    'Net_Growth',
    'Birth_Rate - Death_Rate',
    units='people/year'
)
net_growth.add_dependency(birth_rate)
net_growth.add_dependency(death_rate)

# Validate units
validation = net_growth.validate_units()
print("✅ Valid Calculator:")
print(f"Expression: {net_growth.expression}")
print(f"Expected units: {net_growth.units}")
print(f"Validation: {'PASSED' if validation['valid'] else 'FAILED'}")
print(f"Inferred units: {validation['inferred_units']}")

# Calculate result
result = net_growth.calculate(0, 0.25)
print(f"Result: {result} {net_growth.units}")

## Example 2: Invalid Unit Combinations

In [ ]:
# Create elements with incompatible units
population = Stock('Population', initial_value=1000, units='people')
birth_rate = Flow('Birth_Rate', rate=25, units='people/year')

# Invalid calculator: people + people/year (dimensionally inconsistent)
invalid_calc = Calculator(
    'Invalid_Calc',
    'Population + Birth_Rate',
    units='people'
)
invalid_calc.add_dependency(population)
invalid_calc.add_dependency(birth_rate)

# Validate units
validation = invalid_calc.validate_units()
print("❌ Invalid Calculator:")
print(f"Expression: {invalid_calc.expression}")
print(f"Expected units: {invalid_calc.units}")
print(f"Validation: {'PASSED' if validation['valid'] else 'FAILED'}")

if validation['errors']:
    print("Errors:")
    for error in validation['errors']:
        print(f"  - {error}")

# Try to calculate (will issue warnings)
print("\nAttempting calculation...")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    result = invalid_calc.calculate(0, 0.25)
    print(f"Result: {result} (with {len(w)} warnings)")
    for warning in w:
        print(f"Warning: {warning.message}")

## Example 3: Strict Validation Mode

In [ ]:
# Enable strict validation (raises exceptions instead of warnings)
invalid_calc.set_strict_unit_validation(True)

print("🚨 Strict Validation Mode:")
try:
    result = invalid_calc.calculate(0, 0.25)
    print(f"❌ Calculation unexpectedly succeeded: {result}")
except ValueError as e:
    print(f"✅ Exception correctly raised: {e}")
except Exception as e:
    print(f"❌ Unexpected exception: {e}")

## Example 4: YAML Model Validation

In [ ]:
# Load and validate a YAML model
builder = YAMLSystemBuilder()
model = builder.build_from_file('yaml_models/multidimensional_calculator_example.yaml')

print("📊 YAML Model Calculator Validation:")
print("=" * 50)

# Test specific calculators
calculator_names = [
    'Net_Migration_Flow',
    'Natural_Growth_Flow', 
    'Total_Population_Change',
    'Dependency_Ratio'
]

for calc_name in calculator_names:
    calc = model.get_element_by_name(calc_name)
    if calc:
        validation = calc.validate_units()
        status = "✅ PASSED" if validation['valid'] else "❌ FAILED"
        
        print(f"\n{calc_name}:")
        print(f"  Expression: {calc.expression}")
        print(f"  Units: {calc.units}")
        print(f"  Validation: {status}")
        
        if validation['inferred_units']:
            print(f"  Inferred: {validation['inferred_units']}")
        
        if validation['errors']:
            for error in validation['errors']:
                print(f"  Error: {error}")
        
        if validation['warnings']:
            for warning in validation['warnings']:
                print(f"  Warning: {warning}")

## Example 5: Validation Summary

In [ ]:
# Create a calculator and show validation summary
flow1 = Flow('Flow1', rate=5, units='people/year')
flow2 = Flow('Flow2', rate=3, units='people/year')

calc = Calculator('Test_Calc', 'Flow1 - Flow2', units='people/year')
calc.add_dependency(flow1)
calc.add_dependency(flow2)

# Run validation and show summary
calc.validate_units()
print("📋 Validation Summary:")
print(calc.get_validation_summary())

## Summary

The new unit validation system provides:

### ✅ Features
- **Dimensional Analysis**: Validates that arithmetic operations are dimensionally consistent
- **Unit Inference**: Automatically infers result units from expressions
- **Error Detection**: Catches incompatible unit combinations
- **Flexible Modes**: Warning mode (default) or strict exception mode
- **YAML Integration**: Automatically validates calculators loaded from YAML
- **Clear Messages**: Provides detailed error and warning messages

### 🔧 Usage
- **Enable/Disable**: `calculator.enable_unit_validation(True/False)`
- **Strict Mode**: `calculator.set_strict_unit_validation(True)`
- **Manual Validation**: `calculator.validate_units()`
- **Summary**: `calculator.get_validation_summary()`

### 🎯 Benefits
- **Prevents Errors**: Catches unit mismatches before they cause problems
- **Improves Reliability**: Ensures model calculations are dimensionally sound
- **Better Debugging**: Clear error messages help identify and fix issues
- **Model Quality**: Enforces good modeling practices

The original issue of missing unit validation has been **completely resolved**!